# Derivatives and Differentiation Rules

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 02.01 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/02_derivatives_and_rules.ipynb)

---

## 🎯 Learning Objective

Master the power, chain, product, and quotient rules, and see derivatives as instantaneous rates of sensitivity.

---

## 📐 Theory

The derivative answers a sharper version of the limit question from `02.01`: not just "what
value does $f$ approach," but "how fast is $f$ changing right now." Formally,

$$f'(a) = \lim_{h \to 0} \frac{f(a+h) - f(a)}{h}$$

the limit of average rates of change over shrinking intervals. Geometrically, $f'(a)$ is the
slope of the tangent line at $a$ — the best *linear* approximation to $f$ near that point.
This limit only exists where $f$ behaves well locally, which is why continuity (`02.01`) is
necessary (though not sufficient — $|x|$ is continuous at $0$ but not differentiable there,
since the left and right slopes disagree).

### The rules that make derivatives tractable

Computing a derivative from the limit definition every time would be unworkable, so calculus
gives you a small rule set that composes:

**Power rule:** $\frac{d}{dx} x^n = n x^{n-1}$.

**Sum rule:** $\frac{d}{dx}[f(x) + g(x)] = f'(x) + g'(x)$ — derivatives distribute over sums.

**Product rule:** $\frac{d}{dx}[f(x)g(x)] = f'(x)g(x) + f(x)g'(x)$. Neither term alone
captures the full rate of change — a product changes both because $f$ changes *and* because
$g$ changes, so both contributions must be summed.

**Quotient rule:** $\frac{d}{dx}\left[\frac{f(x)}{g(x)}\right] = \frac{f'(x)g(x) - f(x)g'(x)}{g(x)^2}$,
which follows from the product rule applied to $f(x) \cdot g(x)^{-1}$.

**Chain rule:** $\frac{d}{dx}f(g(x)) = f'(g(x)) \cdot g'(x)$. This is the single most important
rule in this notebook's sequel (`02.04`), because it says the rate of change of a *composition*
is the product of the rates of change of each piece — exactly the mechanism that makes
backpropagation through many stacked layers possible.

### Higher-order derivatives

Differentiating $f'$ again gives $f''$, the **second derivative** — the rate of change of the
rate of change, i.e. curvature. $f''(x) > 0$ means the function curves upward (convex) at $x$;
$f''(x) < 0$ means it curves downward (concave). This generalizes to $f^{(n)}$, and reappears
in Notebook `02.06` as the Hessian once we have more than one input variable, and in
optimization (`04.07`) as the key ingredient of Newton's method.

### Derivatives as sensitivity

Rewriting the definition slightly, for small $h$:

$$f(a + h) \approx f(a) + f'(a) \cdot h$$

$f'(a)$ tells you how much the output moves *per unit* of input perturbation — it's a
sensitivity coefficient. A large $|f'(a)|$ means small input changes cause large output
swings; near $f'(a) \approx 0$, the function is locally flat and insensitive. This
reinterpretation — derivative as sensitivity rather than "slope of a graph" — is exactly the
lens you need once inputs become high-dimensional vectors (weights, embeddings) rather than a
single number, which is where we're headed in the rest of this module.

---

In [ ]:
# Setup
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

The tangent line at $a$ is the line through $(a, f(a))$ with slope $f'(a)$. Zooming in tightly
enough around $a$, the curve and its tangent line become visually indistinguishable — that's
the "best linear approximation" idea made literal.

In [ ]:
# Tangent lines as local linear approximations, plus zooming in to make the point visually
f = lambda x: x ** 3 - 2 * x
fprime = lambda x: 3 * x ** 2 - 2  # derived by hand via the power + sum rules

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.linspace(-2, 2, 400)
for a, ax in zip([-1.2, 0.8], axes):
    slope = fprime(a)
    tangent = f(a) + slope * (x - a)
    ax.plot(x, f(x), color="#4C72B0", lw=2, label=r"$f(x)=x^3-2x$")
    ax.plot(x, tangent, color="#DD8452", lw=1.5, ls="--", label=f"tangent at a={a} (slope={slope:.2f})")
    ax.plot(a, f(a), 'ko', zorder=5)
    ax.set_ylim(-3, 3); ax.set_xlabel("x"); ax.set_ylabel("f(x)")
    ax.set_title(f"Tangent line at a={a}")
    ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

# Zoom in around a=0.8 to show the curve and tangent line become indistinguishable
fig, ax = plt.subplots(figsize=(6, 4.5))
a = 0.8
x_zoom = np.linspace(a - 0.05, a + 0.05, 200)
ax.plot(x_zoom, f(x_zoom), color="#4C72B0", lw=3, label="f(x)")
ax.plot(x_zoom, f(a) + fprime(a) * (x_zoom - a), color="#DD8452", lw=1.5, ls="--", label="tangent")
ax.set_title("Zoomed in near a=0.8: curve and tangent overlap")
ax.legend()
plt.show()

## 🐍 Implementation from Scratch

We implement a finite-difference derivative (the definition, directly), then a tiny symbolic
differentiator that applies the power/sum/product/chain rules to an expression tree — a scaled-
down preview of what SymPy (and PyTorch's autograd) do internally.

In [ ]:
def finite_diff(f, a, h=1e-6):
    # Central difference is more accurate than one-sided: errors cancel to O(h^2) instead of O(h)
    return (f(a + h) - f(a - h)) / (2 * h)

# A tiny symbolic differentiator: expressions are nested tuples, e.g. ('pow', ('var',), 3)
# means x^3. Each rule below is exactly one bullet from the Theory section, implemented literally.
def diff_expr(expr):
    kind = expr[0]
    if kind == 'const':
        return ('const', 0.0)
    if kind == 'var':
        return ('const', 1.0)
    if kind == 'pow':
        _, base, n = expr
        # power rule: d/dx x^n = n * x^(n-1)
        return ('mul', ('const', n), ('pow', base, n - 1))
    if kind == 'add':
        _, f, g = expr
        return ('add', diff_expr(f), diff_expr(g))          # sum rule
    if kind == 'mul':
        _, f, g = expr
        return ('add', ('mul', diff_expr(f), g), ('mul', f, diff_expr(g)))  # product rule

def eval_expr(expr, x):
    kind = expr[0]
    if kind == 'const':
        return expr[1]
    if kind == 'var':
        return x
    if kind == 'pow':
        _, base, n = expr
        return eval_expr(base, x) ** n
    if kind == 'add':
        _, f, g = expr
        return eval_expr(f, x) + eval_expr(g, x)
    if kind == 'mul':
        _, f, g = expr
        return eval_expr(f, x) * eval_expr(g, x)

# Represent f(x) = x^3 - 2x as add(pow(x,3), mul(const(-2), pow(x,1)))
expr = ('add', ('pow', ('var',), 3), ('mul', ('const', -2.0), ('pow', ('var',), 1)))
d_expr = diff_expr(expr)

x_test = 0.8
print(f"f(x)  = x^3 - 2x  at x={x_test}:  hand-rolled eval = {eval_expr(expr, x_test):.6f}")
print(f"f'(x) via symbolic rules      : {eval_expr(d_expr, x_test):.6f}")
print(f"f'(x) via finite differences  : {finite_diff(lambda x: x**3 - 2*x, x_test):.6f}")

# Cross-check every rule against SymPy, the "real" symbolic engine
x = sp.symbols('x')
sympy_deriv = sp.diff(x**3 - 2*x, x)
print(f"f'(x) via SymPy               : {float(sympy_deriv.subs(x, x_test)):.6f}")

# Product rule specifically: d/dx[x^2 * sin(x)] -- verify our rule against SymPy's
lhs = sp.diff(x**2 * sp.sin(x), x)
rhs = 2*x*sp.sin(x) + x**2*sp.cos(x)   # our product rule, applied by hand
print(f"\nProduct rule check -- SymPy agrees our hand-applied rule simplifies to zero difference:",
      sp.simplify(lhs - rhs) == 0)

## 🤖 Why This Matters for AI

Every gradient PyTorch computes during `.backward()` is built from exactly these four rules,
applied automatically and repeatedly. There's no separate "AI calculus" — a transformer's
loss function is just a very long composition of sums, products, powers, and nested functions,
and its gradient is the chain rule applied at industrial scale (the full story is
`02.04`–`02.05`). The "sensitivity" reading of the derivative is also literally how **saliency
maps** work: to see which input pixels/tokens a model's prediction is most sensitive to, you
compute $\partial(\text{output})/\partial(\text{input})$ and visualize its magnitude. Below, we
verify PyTorch's autograd reproduces our hand-derived product-rule result exactly, then build a
minimal saliency map for a toy regression model.

In [ ]:
import torch

# 1) Confirm autograd's product rule matches our hand derivation and SymPy, exactly
x_t = torch.tensor(0.8, requires_grad=True)
y_t = x_t**2 * torch.sin(x_t)
y_t.backward()
hand_derived = 2 * 0.8 * np.sin(0.8) + 0.8**2 * np.cos(0.8)
print(f"autograd d/dx[x^2 sin(x)] at x=0.8 : {x_t.grad.item():.6f}")
print(f"hand-derived product rule          : {hand_derived:.6f}")

# 2) A minimal saliency map: which input features is a toy model's output most sensitive to?
# "Sensitivity" here is EXACTLY the |f'(a)| reading from the Theory section, per input dimension.
torch.manual_seed(0)
# Small weight scale keeps tanh's pre-activation near 0, where tanh' is largest and most
# informative -- with large weights tanh saturates and EVERY feature's saliency collapses to ~0
W = torch.tensor([2.0, 0.1, -1.5, 0.3, 0.05]).reshape(5, 1) * 0.3
features = torch.randn(5, requires_grad=True)
output = torch.tanh(features @ W).squeeze()   # nonlinear, so sensitivity varies by feature
output.backward()

saliency = features.grad.abs()
print("\nInput features:      ", np.round(features.detach().numpy(), 3))
print("Saliency |d(output)/d(feature)|:", np.round(saliency.numpy(), 3))
most_sensitive = saliency.argmax().item()
print(f"Most influential feature: index {most_sensitive} "
      f"(perturbing it changes the output the most, per unit perturbation)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(5), saliency.numpy(), color="#4C72B0")
ax.set_xlabel("input feature index"); ax.set_ylabel("|d(output)/d(feature)|")
ax.set_title("Saliency: derivative-as-sensitivity applied to a model's inputs")
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Using the power, sum, and product rules by hand, differentiate
   $f(x) = 3x^4 - 5x^2 + x \cdot \sin(x)$, then evaluate $f'(1)$. Verify both the symbolic form
   (with `sp.diff`) and the numeric value (with `finite_diff`).

<details>
<summary>💡 Solution</summary>

Power/sum rule on the first two terms: $\frac{d}{dx}[3x^4-5x^2] = 12x^3-10x$. Product rule on
the third term ($u=x$, $v=\sin x$): $\frac{d}{dx}[x\sin x] = \sin x + x\cos x$. Summing,
$$f'(x) = 12x^3 - 10x + \sin x + x\cos x.$$
At $x=1$: $f'(1) = 12 - 10 + \sin(1) + \cos(1) \approx 2 + 0.841471 + 0.540302 = 3.381773$.
`sp.diff` reproduces the same symbolic expression exactly, and `finite_diff(f, 1.0)` gives
$\approx 3.3817732905$ — matching the exact value $3.3817732907$ to about $10^{-10}$, well
within central difference's expected error at $h=10^{-6}$.

</details>

### 💭 UNDERSTAND
2. A common mistake is to assume $\frac{d}{dx}[f(x)g(x)] = f'(x)g'(x)$ — multiplying the two
   derivatives directly, by analogy with how sums differentiate termwise. Using
   $f(x)=x^2$, $g(x)=\sin(x)$ at $x=0.8$, compute both the correct product-rule result and this
   "multiply the derivatives" result, and explain from the product rule's own derivation
   (via the limit definition) why the two don't agree.

<details>
<summary>💡 Solution</summary>

Correct: $\frac{d}{dx}[x^2\sin x] = 2x\sin x + x^2\cos x$, which at $x=0.8$ is
$\approx 1.593662$. The "multiply the derivatives" guess gives $f'(x)g'(x) = 2x\cos x$, which
at $x=0.8$ is $\approx 1.114731$ — a different, wrong value. The reason sums differentiate
termwise but products don't is visible directly in the limit definition: expanding
$\frac{(f(x+h)g(x+h)) - f(x)g(x)}{h}$ and adding and subtracting $f(x)g(x+h)$ produces *two*
terms, $g(x+h)\cdot\frac{f(x+h)-f(x)}{h}$ and $f(x)\cdot\frac{g(x+h)-g(x)}{h}$ — a product
changes because $f$ moves *and* because $g$ moves, and both contributions survive in the limit
(giving $f'g+fg'$), whereas $f'g'$ would only appear if you tried to account for both moving
*at once*, a genuinely higher-order (and vanishingly small, $O(h)$) effect that drops out as
$h\to0$.

</details>

### ✏️ DERIVE
3. The Theory section states the quotient rule "follows from the product rule applied to
   $f(x)\cdot g(x)^{-1}$." Carry out that derivation explicitly: write $h(x)=f(x)g(x)^{-1}$,
   apply the product rule, then apply the chain rule (with the power rule) to
   $\frac{d}{dx}[g(x)^{-1}]$, and simplify to the standard quotient-rule form.

<details>
<summary>💡 Solution outline</summary>

Write $h(x) = f(x)\cdot g(x)^{-1}$. Product rule:
$$h'(x) = f'(x)\,g(x)^{-1} + f(x)\cdot\frac{d}{dx}\big[g(x)^{-1}\big].$$
For the remaining piece, treat $g(x)^{-1}$ as a composition $u^{-1}$ with $u=g(x)$: the chain
rule gives $\frac{d}{dx}[u^{-1}] = -u^{-2}\cdot u'$, and here $u'=g'(x)$ (the power rule
supplies $\frac{d}{du}u^{-1}=-u^{-2}$), so
$$\frac{d}{dx}\big[g(x)^{-1}\big] = -g(x)^{-2}g'(x).$$
Substituting back:
$$h'(x) = \frac{f'(x)}{g(x)} - \frac{f(x)g'(x)}{g(x)^2} = \frac{f'(x)g(x) - f(x)g'(x)}{g(x)^2},$$
exactly the quotient rule stated in the Theory section. This derivation is also why the
quotient rule needs no separate proof or separate entry in `diff_expr`'s rule set in
principle — it's the product and chain rules composed — though IMPLEMENT below adds it as its
own case for convenience rather than rewriting every division as a literal power of $-1$.

</details>

### 🐍 IMPLEMENT
4. Implement the quotient rule as a case in `diff_expr` (represent $f/g$ as `('div', f, g)`,
   returning the DERIVE exercise's formula built from existing AST node types), then verify it
   against SymPy **and** `finite_diff` on $h(x) = \frac{x^2 + 1}{x - 3}$ at $x=5$.

<details>
<summary>✅ How to know you're right</summary>

All three computations of $h'(5)$ — your new `diff_expr` case, `sp.diff`, and `finite_diff` —
should agree to within floating-point/finite-difference tolerance (the exact value is
$h'(5)=-1.5$; `finite_diff` should land within about $10^{-8}$ of it at the default
$h=10^{-6}$). As a directional sanity check: $h(4.9)\approx13.163$, $h(5)=13.0$,
$h(5.1)\approx12.862$ — $h$ is decreasing through $x=5$, consistent with the negative sign of
your computed $h'(5)$. If your quotient-rule case instead gives $+1.5$ or a value far from
$-1.5$, check the sign in the numerator ($f'g-fg'$, not $f'g+fg'$) first — it's the most common
place to drop a minus sign when building this out of `add`/`mul` nodes.

</details>

### 🤖 APPLY
5. Extend the saliency example to a 2-layer network (`Linear(5,8) -> tanh -> Linear(8,1)`).
   Compute saliency for 20 random input vectors, average the absolute saliency per feature
   across all 20, and compare the feature ranking you get from the *average* against the
   ranking from any single input alone.

<details>
<summary>✅ What you should observe</summary>

The most-influential feature according to the 20-input average should be noticeably more
stable than the most-influential feature according to any one input alone. In one run (seed
0), the per-input winning feature split across the 20 draws as 11-of-20 for feature 1, 8-of-20
for feature 4, and 1-of-20 for feature 2 — three *different* features would each look like "the
most important one" depending on which single input you happened to check, while the
20-input average saliency picked feature 1 unambiguously (average saliency
$\approx[0.039, 0.166, 0.093, 0.133, 0.163]$) and matched the plurality winner. This is the
same instability the notebook's `torch.randn` seed already hints at: a saliency map is really a
*local* sensitivity reading (the derivative at one specific point), and a nonlinear model's
sensitivity genuinely varies from input to input — averaging over many inputs is what turns a
single noisy local reading into a more trustworthy statement about which feature the model
relies on *globally*.

</details>

### 🚀 CHALLENGE
6. `finite_diff` uses a **central** difference, $\frac{f(a+h)-f(a-h)}{2h}$. Investigate two of
   its failure/tradeoff modes: (a) at a genuine kink — $f(x)=|x|$ at $a=0$, where the true
   derivative doesn't exist — sweep $h$ from $1$ down to $10^{-12}$ and see what `finite_diff`
   reports; explain algebraically why a *central* difference behaves this way at a kink
   specifically (compare against the one-sided forward and backward differences at the same
   point). (b) On a smooth function ($f(x)=\sin(x)$ at $a=1$, where $f'(a)=\cos(1)$ is known
   exactly), sweep $h$ from $10^{-1}$ down to $10^{-16}$ and plot the absolute error vs. $h$ on
   a log-log scale. Identify the two distinct regimes and the (approximate) $h$ where error is
   minimized.

<details>
<summary>🔍 What a strong answer covers</summary>

For (a): `finite_diff(abs, 0.0, h)` returns *exactly* $0.0$ for every $h$ tested, from $1$ down
to $10^{-12}$ — never erroring, never wavering, and never revealing that $|x|$ isn't
differentiable at $0$. This isn't a bug or numerical coincidence: a central difference at a
symmetric kink always computes the *average* of the left- and right-hand slopes, since
$|{+h}|=|{-h}|=h$ exactly cancels the asymmetry that would otherwise show up; more generally
(shown on an asymmetric kink with left slope $-2$ and right slope $4$), the central difference
returns exactly $(-2+4)/2=1$ regardless of $h$, while the one-sided forward and backward
differences correctly report the *different* one- and other-sided slopes ($4$ and $-2$
respectively) and disagree with each other — that disagreement is the real signal that the
derivative doesn't exist, which the central difference's averaging silently erases. For (b):
error should decrease roughly linearly on the log-log plot as $h$ shrinks from $10^{-1}$ (error
$\approx9\times10^{-4}$) down to around $h\approx10^{-5}$ (error $\approx10^{-11}$) — this is
$O(h^2)$ truncation error dominating — then *increase* again as $h$ keeps shrinking past that
point, reaching error $\approx10^{-2}$ by $h=10^{-15}$–$10^{-16}$, because subtracting two
nearly-equal floating-point numbers ($f(a+h)$ and $f(a-h)$) loses relative precision
(catastrophic cancellation) once $h$ is small enough that the difference is comparable to
floating-point round-off. A strong answer names both regimes explicitly (truncation error
shrinking vs. round-off error growing) and connects the existence of a sweet-spot $h$ (here
around $10^{-5}$–$10^{-6}$) to why real automatic-differentiation systems (autograd, `02.04`
onward) avoid finite differences for computing gradients in the first place — they use exact
symbolic/algorithmic rules with no $h$ to tune and no cancellation error at all.

</details>

---

## 📚 Further Reading
- Stewart, *Calculus*, Ch. 3 (Differentiation Rules)
- Simonyan, Vedaldi & Zisserman, ["Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps"](https://arxiv.org/abs/1312.6034)
- Paszke et al., ["Automatic Differentiation in PyTorch"](https://openreview.net/pdf?id=BJJsrmfCZ)

---

*Next notebook: [Partial Derivatives](03_partial_derivatives.ipynb)*